# 01 — Exploração (EDA)

**Objetivo:** entender o dataset antes de qualquer modelagem.  
Perguntas que este notebook responde:
1. Qual o volume total e distribuição por sub-fluxo?
2. Qual a taxa de resolução geral e por sub-fluxo?
3. Qual a distribuição de fricção (try-again, loops de verificação)?
4. Quais sub-fluxos têm maior impacto potencial?

## 0. Setup

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR   = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download do Dataset (executar uma vez)

In [ ]:
import urllib.request, gzip, shutil, os

ABCD_URL  = "https://github.com/asappresearch/abcd/raw/master/data/abcd_v1.1.json.gz"
ABCD_GZ   = RAW_DIR / "abcd_v1.1.json.gz"
ABCD_JSON = RAW_DIR / "abcd_v1.1.json"

if not ABCD_JSON.exists():
    print("Baixando...")
    urllib.request.urlretrieve(ABCD_URL, ABCD_GZ)
    with gzip.open(ABCD_GZ, 'rb') as fi, open(ABCD_JSON, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    print(f"OK — {ABCD_JSON.stat().st_size/1e6:.0f} MB")
else:
    print(f"Já existe — {ABCD_JSON.stat().st_size/1e6:.0f} MB")

## 2. Carregamento e Parsing

### Decisão de design: definição de resolução e fricção

O ABCD não tem rótulo de resolução nativo. Engenheiramos dois sinais:

| Sinal | Definição | Justificativa |
|---|---|---|
| `resolution=0` | Última ação = `notify-team` E subflow ≠ `out_of_stock_general` | Escalação inesperada: o agente não conseguiu resolver sem intervenção humana |
| `has_friction=1` | Conversa tem `try-again` OU `verify-identity` repetido >1x | Indicador de que a solução inicial falhou e houve retrabalho |

In [ ]:
with open(ABCD_JSON, 'r', encoding='utf-8') as f:
    raw = json.load(f)

records = []
for split, convos in raw.items():
    for c in convos:
        sf   = c['scenario']['subflow']
        fl   = c['scenario']['flow']
        turns = c['original']
        all_actions = [t['targets'][2] for t in c['delexed']
                       if t['targets'][1] == 'take_action']
        last_action = all_actions[-1] if all_actions else None

        is_escalation = (last_action == 'notify-team' and sf != 'out_of_stock_general')
        resolution    = 0 if is_escalation else 1

        try_again_n = all_actions.count('try-again')
        verify_n    = all_actions.count('verify-identity')
        has_loop    = int(verify_n > 1)
        has_friction = int(try_again_n > 0 or has_loop)

        records.append({
            'convo_id'        : c['convo_id'],
            'split'           : split,
            'flow'            : fl,
            'subflow'         : sf,
            'n_turns'         : len(turns),
            'n_turns_customer': len([t for t in turns if t[0]=='customer']),
            'n_turns_agent'   : len([t for t in turns if t[0]=='agent']),
            'n_actions'       : len(all_actions),
            'n_try_again'     : try_again_n,
            'n_verify'        : verify_n,
            'last_action'     : last_action,
            'resolution'      : resolution,
            'has_try_again'   : int(try_again_n > 0),
            'has_loop'        : has_loop,
            'has_friction'    : has_friction,
        })

df = pd.DataFrame(records)
print(f"Total: {len(df):,} conversas | {df['subflow'].nunique()} subflows")
df.head(3)

## 3. Volume e Qualidade

In [ ]:
print(f"Missings:\n{df.isnull().sum()[df.isnull().sum()>0]}")
print(f"\nconvo_id únicos: {df['convo_id'].nunique():,}")

print("\n=== Volume por split ===")
print(df.groupby('split').agg(n=('convo_id','count'), resolucao=('resolution','mean')).round(3))

## 4. Métricas Globais

In [ ]:
print("=== RESOLUÇÃO ===")
print(f"  Taxa geral:            {df['resolution'].mean():.1%}")
print(f"  Escalações:            {(1-df['resolution']).sum():.0f} conversas")

print("\n=== FRICÇÃO ===")
print(f"  Has try-again:         {df['has_try_again'].mean():.1%} ({df['has_try_again'].sum():,} conversas)")
print(f"  Has loop (verify >1x): {df['has_loop'].mean():.1%} ({df['has_loop'].sum():,} conversas)")
print(f"  Any friction:          {df['has_friction'].mean():.1%} ({df['has_friction'].sum():,} conversas)")

print("\n=== COMPRIMENTO ===")
print(df['n_turns'].describe().round(1))

## 5. Distribuição por Flow

In [ ]:
flow_stats = (
    df.groupby('flow')
    .agg(n=('convo_id','count'),
         resolucao=('resolution','mean'),
         friccao=('has_friction','mean'))
    .sort_values('friccao', ascending=False)
    .reset_index()
)
flow_stats[['resolucao','friccao']] = (flow_stats[['resolucao','friccao']] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(flow_stats['flow'], flow_stats['n'], color='steelblue', edgecolor='none')
axes[0].set_xlabel('Nº de conversas')
axes[0].set_title('Volume por Flow')

colors = ['#d62728' if v > 10 else '#aec7e8' for v in flow_stats['friccao']]
axes[1].barh(flow_stats['flow'], flow_stats['friccao'], color=colors, edgecolor='none')
axes[1].axvline(df['has_friction'].mean()*100, color='navy', ls='--', lw=1.5,
                label=f"Média ({df['has_friction'].mean():.0%})")
axes[1].set_xlabel('Taxa de Fricção (%)')
axes[1].set_title('Fricção por Flow')
axes[1].legend()

plt.suptitle('ABCD Dataset — Visão por Flow', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_flows_overview.png', bbox_inches='tight')
plt.show()

print(flow_stats.to_string(index=False))

## 6. Drill-down: troubleshoot_site

O flow `troubleshoot_site` tem **taxa de fricção 6-7x acima da média**. Vamos entender por subflow.

In [ ]:
ts = df[df['flow'] == 'troubleshoot_site'].copy()

ts_sf = (
    ts.groupby('subflow')
    .agg(n=('convo_id','count'),
         resolucao=('resolution','mean'),
         friccao=('has_friction','mean'),
         try_again=('has_try_again','mean'),
         turnos_med=('n_turns','mean'))
    .sort_values('friccao', ascending=False)
    .reset_index()
)
ts_sf[['resolucao','friccao','try_again']] = (ts_sf[['resolucao','friccao','try_again']]*100).round(1)
ts_sf['turnos_med'] = ts_sf['turnos_med'].round(1)
print(ts_sf.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = ts_sf['subflow']
w = 0.35
pos = range(len(x))

bars1 = ax.bar([p - w/2 for p in pos], ts_sf['friccao'],  width=w, label='Fricção (%)', color='#d62728')
bars2 = ax.bar([p + w/2 for p in pos], ts_sf['resolucao'], width=w, label='Resolução (%)', color='#2ca02c')

ax.set_xticks(list(pos))
ax.set_xticklabels(x, rotation=15, ha='right')
ax.set_ylabel('%')
ax.set_title('troubleshoot_site — Fricção vs Resolução por Subflow', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_troubleshoot_site_detail.png', bbox_inches='tight')
plt.show()

## 7. Distribuição de Comprimento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['n_turns'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['n_turns'].mean(), color='red', ls='--', label=f"Média = {df['n_turns'].mean():.0f}")
axes[0].set_xlabel('Número de turnos')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de comprimento')
axes[0].legend()

for friction_val, label, color in [(0, 'Sem fricção', '#2ca02c'), (1, 'Com fricção', '#d62728')]:
    subset = df[df['has_friction'] == friction_val]['n_turns']
    axes[1].hist(subset, bins=30, alpha=0.6, label=f'{label} (n={len(subset):,})', color=color)
axes[1].set_xlabel('Número de turnos')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Comprimento: com vs sem fricção')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_comprimento_conversas.png', bbox_inches='tight')
plt.show()

print(f"Média turnos (sem fricção): {df[df['has_friction']==0]['n_turns'].mean():.1f}")
print(f"Média turnos (com fricção): {df[df['has_friction']==1]['n_turns'].mean():.1f}")

## 8. Preview do ICE Ranking (antecipação do notebook 05)

In [ ]:
sf_stats = (
    df.groupby(['flow','subflow'])
    .agg(n=('convo_id','count'),
         friccao=('has_friction','mean'),
         resolucao=('resolution','mean'))
    .reset_index()
)
sf_stats['impact_raw'] = sf_stats['n'] * sf_stats['friccao']
sf_stats[['friccao','resolucao']] = (sf_stats[['friccao','resolucao']]*100).round(1)
sf_stats['impact_raw'] = sf_stats['impact_raw'].round(0)

top10 = sf_stats.nlargest(10, 'impact_raw')[['flow','subflow','n','friccao','resolucao','impact_raw']]
print("Top 10 subflows por impacto bruto (n × fricção):")
print(top10.to_string(index=False))

## 9. Exportar Base Processada

In [ ]:
df.to_parquet(PROCESSED_DIR / 'conversations_base.parquet', index=False)
sf_stats.to_csv(PROCESSED_DIR / 'subflow_stats.csv', index=False)
print(f"Salvo: {len(df):,} conversas | {df['subflow'].nunique()} subflows")

## 10. Sumário da EDA

| Indicador | Valor |
|---|---|
| Total de conversas | 10.042 |
| Splits | train 8.034 / dev 1.004 / test 1.004 |
| Sub-fluxos únicos | 96 |
| Taxa de resolução geral | 98,0% |
| Escalações (não resolvidas) | 198 |
| Fricção geral | 9,2% das conversas |
| Média de turnos por conversa | 22 |
| Média de turnos (com fricção) | ~26 |
| Flow com maior fricção | **troubleshoot_site: 66,2%** |
| Subflow #1 ICE | **shopping_cart: 84% fricção, 251 conversas** |

**Achado principal:** `troubleshoot_site` concentra 98% do impacto bruto de fricção. Dentro desse flow, `shopping_cart` e `credit_card` têm taxas de try-again acima de 78% — o agente quase sempre precisa pedir ao cliente que tente novamente após uma falha inicial.

**Próximo passo:** notebook 02 — análise de padrões de falha nas conversas de alto atrito.